# 🇨🇴 Colombia INS (Instituto Nacional de Salud) Data Access

This notebook demonstrates how to access public health surveillance data from **Colombia's National Health Institute (INS)**, including SIVIGILA (Public Health Surveillance System) data.

## Overview

The Colombian INS manages the SIVIGILA system for notifiable disease surveillance, covering:
- **Vector-borne diseases**: Dengue, Malaria, Chikungunya, Zika, Yellow fever
- **Respiratory diseases**: COVID-19, Influenza, IRA
- **Water and food-borne diseases**: Hepatitis, EDA
- **Sexually transmitted infections**: VIH/SIDA, Syphilis
- **Tuberculosis and Leprosy**
- **Public health events of interest**: Suicide attempts, violence, traffic accidents

**Website:** https://www.ins.gov.co/

**Open Data Portal:** https://www.datos.gov.co

## 📦 Setup

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

from epidatasets.sources.colombia_ins import ColombiaINSAccessor

# Initialize accessor
ins = ColombiaINSAccessor()

print("✅ Colombia INS Accessor initialized successfully!")

✅ Colombia INS Accessor initialized successfully!


## 🗺️ Exploring Colombian Geography

Colombia is divided into 33 departments (departamentos) organized into 5 geographical regions:

In [2]:
# List all departments
departments = ins.list_departments()
print(f"Total departments: {len(departments)}")
print("\nFirst 10 departments:")
departments.head(10)

Total departments: 33

First 10 departments:


,department_code,department_name
0,05,Antioquia
1,08,Atlántico
2,11,"Bogotá, D.C."
3,13,Bolívar
4,15,Boyacá
5,17,Caldas
6,18,Caquetá
7,19,Cauca
8,20,Cesar
9,23,Córdoba


In [3]:
# View departments by region
regions = {
    "Andina": ins.get_departments_by_region('Andina'),
    "Caribe": ins.get_departments_by_region('Caribe'),
    "Pacífica": ins.get_departments_by_region('Pacífica'),
    "Orinoquía": ins.get_departments_by_region('Orinoquía'),
    "Amazonía": ins.get_departments_by_region('Amazonía'),
}

for region, depts in regions.items():
    dept_names = [ins.DEPARTMENTS[d] for d in depts]
    print(f"\n📍 {region} ({len(depts)} departments):")
    print("  " + ", ".join(dept_names[:5]) + ("..." if len(dept_names) > 5 else ""))


📍 Andina (10 departments):
  Antioquia, Bogotá, D.C., Boyacá, Caldas, Cundinamarca...

📍 Caribe (8 departments):
  Atlántico, Bolívar, Cesar, Córdoba, La Guajira...

📍 Pacífica (4 departments):
  Cauca, Chocó, Nariño, Valle del Cauca

📍 Orinoquía (4 departments):
  Meta, Arauca, Casanare, Vichada

📍 Amazonía (6 departments):
  Caquetá, Putumayo, Amazonas, Guainía, Guaviare...


## 🦠 Notifiable Diseases (SIVIGILA)

SIVIGILA tracks 33 notifiable diseases organized into event groups:

In [4]:
# List all diseases
diseases = ins.list_diseases()
print(f"Total notifiable diseases: {len(diseases)}\n")
diseases.head(10)

Total notifiable diseases: 33



,disease_code,disease_name
0,100,DENGUE
1,110,DENGUE GRAVE
2,111,DENGUE CON SIGNOS DE ALARMA
3,120,MALARIA
4,130,CHIKUNGUNYA
5,140,ZIKA
6,150,FIEBRE AMARILLA
7,160,LEISHMANIASIS
8,170,LEPTOSPIROSIS
9,180,RABIA HUMANA


In [5]:
# View diseases by event group
event_groups = ins.list_event_groups()

for group in event_groups['event_group'].unique():
    group_diseases = event_groups[event_groups['event_group'] == group]
    print(f"\n📋 {group} ({len(group_diseases)} diseases):")
    for _, row in group_diseases.head(5).iterrows():
        print(f"  - {row['disease_code']}: {row['disease_name']}")
    if len(group_diseases) > 5:
        print(f"  ... and {len(group_diseases) - 5} more")


📋 ENFERMEDADES TRANSMITIDAS POR VECTORES (9 diseases):
  - 100: DENGUE
  - 110: DENGUE GRAVE
  - 111: DENGUE CON SIGNOS DE ALARMA
  - 120: MALARIA
  - 130: CHIKUNGUNYA
  ... and 4 more

📋 ENFERMEDADES DE TRANSMISIÓN RESPIRATORIA (6 diseases):
  - 310: INFLUENZA
  - 320: IRA (Infecciones Respiratorias Agudas)
  - 350: SARAMPIÓN
  - 360: RUBÉOLA
  - 390: PERTUSSIS
  ... and 1 more

📋 ENFERMEDADES DE TRANSMISIÓN HÍDRICA Y ALIMENTARIA (5 diseases):
  - 240: HEPATITIS B
  - 250: HEPATITIS C
  - 330: EDA (Enfermedades Diarreicas Agudas)
  - 370: TÉTANOS NEONATAL
  - 380: TÉTANOS ACCIDENTAL

📋 TUBERCULOSIS Y LEPRA (1 diseases):
  - 190: TUBERCULOSIS

📋 VIH E ITS (3 diseases):
  - 210: SÍFILIS GESTACIONAL
  - 220: SÍFILIS CONGENITA
  - 230: VIH/SIDA

📋 RABIA (1 diseases):
  - 180: RABIA HUMANA

📋 EVENTOS DE INTERÉS EN SALUD PÚBLICA (4 diseases):
  - 260: INTENTO DE SUICIDIO
  - 270: VIOLENCIA INTERPERSONAL
  - 280: ACCIDENTE DE TRÁNSITO
  - 340: PARALISIS FLÁCIDA AGUDA


## 🦟 Dengue Surveillance

Dengue is a priority disease in Colombia with mandatory weekly reporting. Let's access dengue data:

In [6]:
# Get dengue data for recent years
# Note: Actual data requires integration with datos.gov.co API
dengue_data = ins.get_dengue_data(
    years=[2022, 2023],
    departments=['05', '76', '11']  # Antioquia, Valle del Cauca, Bogotá
)

print(f"Dengue data records: {len(dengue_data)}")
print("\nDengue types tracked:")
print(dengue_data[['disease_code', 'disease_name']].drop_duplicates())

INFO:accessors.colombia_ins:Fetching dengue data for years [2022, 2023], departments ['05', '76', '11']
INFO:accessors.colombia_ins:Fetching SIVIGILA data for 3 diseases, 3 departments


Dengue data records: 18

Dengue types tracked:
   disease_code                 disease_name
0           100                       DENGUE
6           110                 DENGUE GRAVE
12          111  DENGUE CON SIGNOS DE ALARMA


In [7]:
# Dengue by department
print("\nDengue surveillance by department:")
dengue_by_dept = dengue_data.groupby('department_name').size().reset_index(name='records')
dengue_by_dept


Dengue surveillance by department:


,department_name,records
0,Antioquia,6
1,"Bogotá, D.C.",6
2,Valle del Cauca,6


## 🦟 Malaria Surveillance

Malaria is endemic in several Colombian regions, particularly the Pacific coast and Amazon:

In [8]:
# Get malaria data for Amazon and Pacific regions
amazon_depts = ins.get_departments_by_region('Amazonía')
pacific_depts = ins.get_departments_by_region('Pacífica')

malaria_data = ins.get_malaria_data(
    years=[2022, 2023],
    departments=amazon_depts + pacific_depts
)

print(f"Malaria data records: {len(malaria_data)}")
print("\nMalaria types tracked:")
print(malaria_data[['disease_code', 'disease_name']].drop_duplicates())

INFO:accessors.colombia_ins:Fetching malaria data for years [2022, 2023], departments ['18', '86', '91', '94', '95', '97', '19', '27', '52', '76']
INFO:accessors.colombia_ins:Fetching SIVIGILA data for 2 diseases, 10 departments


Malaria data records: 40

Malaria types tracked:
   disease_code        disease_name
0           120             MALARIA
20          290  MALARIA COMPLICADA


## 🦠 Arbovirus Surveillance (Dengue, Chikungunya, Zika, Yellow Fever)

Colombia conducts integrated surveillance for arboviruses due to similar transmission patterns:

In [9]:
# Get all arbovirus data
arbovirus_data = ins.get_arbovirus_data(
    years=[2023],
    departments=ins.get_departments_by_region('Caribe')
)

print(f"Arbovirus surveillance records: {len(arbovirus_data)}")
print("\nArboviruses tracked:")
arbovirus_summary = arbovirus_data[['disease_code', 'disease_name']].drop_duplicates()
arbovirus_summary

INFO:accessors.colombia_ins:Fetching arbovirus data for years [2023], departments ['08', '13', '20', '23', '44', '47', '70', '88']
INFO:accessors.colombia_ins:Fetching SIVIGILA data for 6 diseases, 8 departments


Arbovirus surveillance records: 48

Arboviruses tracked:


,disease_code,disease_name
0,100,DENGUE
8,110,DENGUE GRAVE
16,111,DENGUE CON SIGNOS DE ALARMA
24,130,CHIKUNGUNYA
32,140,ZIKA
40,150,FIEBRE AMARILLA


## 📊 Weekly Epidemiological Bulletins

The INS publishes weekly epidemiological bulletins (Boletines Epidemiológicos Semanales):

In [10]:
# Get bulletins for a specific year
bulletins_2023 = ins.get_weekly_bulletins(year=2023)

print(f"Total bulletins in 2023: {len(bulletins_2023)}")
print("\nFirst 5 weeks:")
bulletins_2023.head()[['year', 'week', 'data_source', 'note']]

INFO:accessors.colombia_ins:Fetching weekly bulletins for year 2023, week all


Total bulletins in 2023: 53

First 5 weeks:


,year,week,data_source,note
0,2023,1,INS Colombia,Bulletins are PDF documents. Data extraction r...
1,2023,2,INS Colombia,Bulletins are PDF documents. Data extraction r...
2,2023,3,INS Colombia,Bulletins are PDF documents. Data extraction r...
3,2023,4,INS Colombia,Bulletins are PDF documents. Data extraction r...
4,2023,5,INS Colombia,Bulletins are PDF documents. Data extraction r...


In [11]:
# Get specific week bulletin
week_10 = ins.get_weekly_bulletins(year=2023, week=10)
print("Week 10 bulletin:")
week_10[['year', 'week', 'bulletin_url', 'diseases_covered']]

INFO:accessors.colombia_ins:Fetching weekly bulletins for year 2023, week 10


Week 10 bulletin:


,year,week,bulletin_url,diseases_covered
0,2023,10,https://www.ins.gov.co/Direcciones/Vigilancia/...,"[DENGUE, DENGUE GRAVE, DENGUE CON SIGNOS DE AL..."


## 📈 Comparing Departments

Compare disease occurrence across departments over time:

In [12]:
# Compare dengue across major cities
major_cities = ['05', '76', '11', '08', '66']  # Antioquia, Valle, Bogotá, Atlántico, Risaralda

comparison = ins.compare_departments(
    disease='100',  # Dengue
    years=[2022, 2023],
    departments=major_cities
)

print("Dengue comparison structure (sample):")
print(comparison.head() if not comparison.empty else "No data available")

INFO:accessors.colombia_ins:Fetching SIVIGILA data for 1 diseases, 5 departments


Dengue comparison structure (sample):
department_code 05 08 11 66 76
year                          
2022             0  0  0  0  0
2023             0  0  0  0  0


## 📋 Summary by Department

Get summary statistics for a specific year:

In [13]:
# Get summary for vector-borne diseases
summary = ins.get_summary_by_department(
    year=2023,
    disease_group='ENFERMEDADES TRANSMITIDAS POR VECTORES'
)

print(f"Summary records: {len(summary)}")
print("\nFirst 10 departments:")
summary.head(10) if not summary.empty else "No data available"

INFO:accessors.colombia_ins:Fetching SIVIGILA data for 9 diseases, all departments


Summary records: 33

First 10 departments:


,department_code,total_cases,total_deaths,n_diseases,department_name,year
0,05,0,0,9,Antioquia,2023
1,08,0,0,9,Atlántico,2023
2,11,0,0,9,"Bogotá, D.C.",2023
3,13,0,0,9,Bolívar,2023
4,15,0,0,9,Boyacá,2023
5,17,0,0,9,Caldas,2023
6,18,0,0,9,Caquetá,2023
7,19,0,0,9,Cauca,2023
8,20,0,0,9,Cesar,2023
9,23,0,0,9,Córdoba,2023


## 🔍 Data Access Notes

### Current Implementation

This accessor provides:
- ✅ Complete metadata (departments, diseases, event groups)
- ✅ Structured data access methods
- ✅ Regional and temporal filtering
- ✅ Ready-to-use data structures

### Future Enhancements

For full data access, integrate with:
1. **datos.gov.co API** - Colombia's open data portal
2. **INS SIVIGILA API** - Direct access to surveillance system
3. **PDF parsing** - Extract data from weekly bulletins

### Data Availability

| Data Type | Frequency | Access Method |
|-----------|-----------|---------------|
| Weekly bulletins | Weekly | PDF downloads |
| Case data | Weekly | datos.gov.co API |
| Annual summaries | Yearly | INS reports |
| Geographic data | Static | Administrative divisions |


## 💾 Export Data

Save processed data for further analysis:

In [ ]:
# Create output directory
output_dir = Path('./output')
output_dir.mkdir(exist_ok=True)

# Export departments
departments.to_csv(output_dir / 'colombia_departments.csv', index=False)
print("✅ Exported: colombia_departments.csv")

# Export diseases
diseases.to_csv(output_dir / 'colombia_diseases.csv', index=False)
print("✅ Exported: colombia_diseases.csv")

# Export event groups
event_groups.to_csv(output_dir / 'colombia_event_groups.csv', index=False)
print("✅ Exported: colombia_event_groups.csv")

print(f"\n📁 All files saved to: {output_dir.absolute()}")

## 📚 References

- **INS Colombia**: https://www.ins.gov.co/
- **SIVIGILA**: https://www.sivigila.gov.co
- **Datos Abiertos Colombia**: https://www.datos.gov.co
- **Event Groups Classification**: Resolución 2113 de 2021 (Colombia)

---

**Last Updated:** March 2026  
**Data Source:** Instituto Nacional de Salud (INS) - Colombia